## **Question 1 Hypothesis Test**

In [4]:
import pandas as pd
import numpy as np
from math import sqrt
from statistics import NormalDist

MODEL_YEAR_COL = "Model Year"

N_RECENT = 5

# hypothesis test baseline
P0 = 0.050

EXCLUDE_YEAR = 2024

In [5]:
# Two sets of data: one with 2024, one without.
df = pd.read_csv("C:/Users/15386\OneDrive - UBC\Downloads\EV_analysis\data\wa_data_cleaned.csv")

df_my = df.copy()
df_my[MODEL_YEAR_COL] = pd.to_numeric(df_my[MODEL_YEAR_COL], errors="coerce").astype("Int64")

df_with = df_my.dropna(subset=[MODEL_YEAR_COL]).copy()
df_wo = df_with[df_with[MODEL_YEAR_COL] < EXCLUDE_YEAR].copy()

print("N (with 2024):", len(df_with))
print("N (without 2024):", len(df_wo))

N (with 2024): 177477
N (without 2024): 170405


In [6]:
def compute_recent_share(df_in, n_recent=5):
    y = df_in[MODEL_YEAR_COL].dropna().astype(int)
    max_year = int(y.max())
    cutoff = max_year - (n_recent - 1)
    is_new = (y >= cutoff)
    return {
        "max_year": max_year,
        "cutoff_year": cutoff,
        "p_hat": float(is_new.mean()),
        "n": int(is_new.shape[0]),
        "k": int(is_new.sum())
    }

res_with = compute_recent_share(df_with, N_RECENT)
res_wo   = compute_recent_share(df_wo, N_RECENT)

res_with, res_wo

({'max_year': 2024,
  'cutoff_year': 2020,
  'p_hat': 0.6936335412475982,
  'n': 177477,
  'k': 123104},
 {'max_year': 2023,
  'cutoff_year': 2019,
  'p_hat': 0.7449605351955635,
  'n': 170405,
  'k': 126945})

In [7]:
#One sample proportion z_test
def prop_ztest_one_sided(k, n, p0=0.5):
    # z = (p_hat - p0) / sqrt(p0*(1-p0)/n)
    p_hat = k / n
    se0 = sqrt(p0 * (1 - p0) / n)
    z = (p_hat - p0) / se0
    p_value = 1 - NormalDist().cdf(z)  # one-sided upper tail
    return float(z), float(p_value)

z_with, p_with = prop_ztest_one_sided(res_with["k"], res_with["n"], P0)
z_wo,   p_wo   = prop_ztest_one_sided(res_wo["k"], res_wo["n"], P0)

print("WITH 2024:", "cutoff>=", res_with["cutoff_year"], "p_hat=", f"{res_with['p_hat']:.3f}", "z=", f"{z_with:.2f}", "p=", p_with)
print("WO 2024:  ", "cutoff>=", res_wo["cutoff_year"],   "p_hat=", f"{res_wo['p_hat']:.3f}",   "z=", f"{z_wo:.2f}",   "p=", p_wo)


WITH 2024: cutoff>= 2020 p_hat= 0.694 z= 1244.12 p= 0.0
WO 2024:   cutoff>= 2019 p_hat= 0.745 z= 1316.30 p= 0.0


In [8]:
# Find 95% confidence interval
def wilson_ci(k, n, alpha=0.05):
    # Wilson score interval
    z = NormalDist().inv_cdf(1 - alpha/2)
    p_hat = k / n
    denom = 1 + (z**2)/n
    center = (p_hat + (z**2)/(2*n)) / denom
    half = (z * sqrt((p_hat*(1-p_hat)/n) + (z**2)/(4*n**2))) / denom
    return float(center - half), float(center + half)

ci_with = wilson_ci(res_with["k"], res_with["n"])
ci_wo   = wilson_ci(res_wo["k"],   res_wo["n"])

print("WITH 2024  CI95%:", tuple(round(x, 3) for x in ci_with))
print("WO 2024    CI95%:", tuple(round(x, 3) for x in ci_wo))

WITH 2024  CI95%: (0.691, 0.696)
WO 2024    CI95%: (0.743, 0.747)


In [9]:
out = pd.DataFrame([
    {
        "dataset": "with_2024",
        "n": res_with["n"],
        "max_year_used": res_with["max_year"],
        "new_definition": f"last_{N_RECENT}_model_years (>= {res_with['cutoff_year']})",
        "p_hat_new_share": res_with["p_hat"],
        "ci95_low": ci_with[0],
        "ci95_high": ci_with[1],
        "z_stat": z_with,
        "p_value_one_sided": p_with,
        "p0_baseline": P0
    },
    {
        "dataset": "exclude_2024",
        "n": res_wo["n"],
        "max_year_used": res_wo["max_year"],
        "new_definition": f"last_{N_RECENT}_model_years (>= {res_wo['cutoff_year']})",
        "p_hat_new_share": res_wo["p_hat"],
        "ci95_low": ci_wo[0],
        "ci95_high": ci_wo[1],
        "z_stat": z_wo,
        "p_value_one_sided": p_wo,
        "p0_baseline": P0
    }
])

out

,dataset,n,max_year_used,new_definition,p_hat_new_share,ci95_low,ci95_high,z_stat,p_value_one_sided,p0_baseline
0,with_2024,177477,2024,last_5_model_years (>= 2020),0.693634,0.691485,0.695774,1244.121808,0.0,0.05
1,exclude_2024,170405,2023,last_5_model_years (>= 2019),0.744961,0.742885,0.747025,1316.298859,0.0,0.05


### Conclusion (Q1 — Model Year concentration)

The hypothesis test results show that the EV registration population in this dataset is highly concentrated in newer model years. Using the most recent 5 model years as the definition of “new,” the estimated share of “new” vehicles is 69.36% for model years ≥ 2020 (95% CI: 69.15%–69.58%) and 74.50% for model years ≥ 2019 (95% CI: 74.29%–74.70%). In both definitions, the one-sided proportion test produces an extremely small p-value (shown as 0 due to numerical precision), so we reject the null hypothesis and conclude that the fleet is significantly skewed toward recent model years.

## **Question 2 Hypothesis Test**

In [10]:
MODEL_YEAR_COL = "Model Year"
EV_TYPE_COL = "Electric Vehicle Type"

# Use your cleaned Q1 data
df_q2 = df_my.copy()

if "ev_type_simplified" not in df_q2.columns:
    def map_ev_type(x):
        if pd.isna(x):
            return "Unknown"
        s = str(x).strip().upper()
        if "BATTERY" in s or s == "BEV":
            return "BEV"
        if "PLUGIN" in s or "PLUG-IN" in s or s == "PHEV":
            return "PHEV"
        return "Other"
    df_q2["ev_type_simplified"] = df_q2[EV_TYPE_COL].apply(map_ev_type)

# Keep only BEV/PHEV for hypothesis tests
df_q2 = df_q2[df_q2["ev_type_simplified"].isin(["BEV", "PHEV"])].copy()

print(df_q2["ev_type_simplified"].value_counts())
print("N total (BEV+PHEV):", len(df_q2))

ev_type_simplified
BEV     138948
PHEV     38529
Name: count, dtype: int64
N total (BEV+PHEV): 177477


In [11]:
def prop_ztest_one_sided(k, n, p0=0.5):
    p_hat = k / n
    se0 = sqrt(p0 * (1 - p0) / n)
    z = (p_hat - p0) / se0
    p_value = 1 - NormalDist().cdf(z)  # upper-tail one-sided
    return float(p_hat), float(z), float(p_value)

def wilson_ci(k, n, alpha=0.05):
    z = NormalDist().inv_cdf(1 - alpha/2)
    p_hat = k / n
    denom = 1 + (z**2)/n
    center = (p_hat + (z**2)/(2*n)) / denom
    half = (z * sqrt((p_hat*(1-p_hat)/n) + (z**2)/(4*n**2))) / denom
    return float(center - half), float(center + half)

n = len(df_q2)
k_bev = int((df_q2["ev_type_simplified"] == "BEV").sum())

p_hat, z, pval = prop_ztest_one_sided(k_bev, n, p0=0.5)
ci_low, ci_high = wilson_ci(k_bev, n)

print("Overall BEV share p̂:", p_hat)
print("95% CI:", (ci_low, ci_high))
print("z-stat:", z)
print("one-sided p-value:", pval, "(may show 0.0 if extremely small)")

Overall BEV share p̂: 0.7829070809175274
95% CI: (0.7809829403448003, 0.7848189748061336)
z-stat: 238.36629861898342
one-sided p-value: 0.0 (may show 0.0 if extremely small)


In [12]:
# Exclude 2024 for main test due to data cutoff (through March 2024)
df_byyear = df_q2[df_q2[MODEL_YEAR_COL] < 2024].copy()

# You can tune bins; this set usually works well.
bins = [1990, 2009, 2014, 2018, 2021, 2023]  # last bin ends at 2023
labels = ["<=2009", "2010-2014", "2015-2018", "2019-2021", "2022-2023"]

df_byyear["year_bin"] = pd.cut(df_byyear[MODEL_YEAR_COL].astype(int), bins=bins, labels=labels, include_lowest=True)

df_byyear["year_bin"].value_counts(dropna=False)

year_bin
2022-2023    85229
2019-2021    41716
2015-2018    33117
2010-2014    10307
<=2009          36
Name: count, dtype: int64

In [13]:
from scipy.stats import chi2_contingency

# contingency table
ct = pd.crosstab(df_byyear["year_bin"], df_byyear["ev_type_simplified"])
ct

ev_type_simplified,BEV,PHEV
year_bin,,
<=2009,36,0
2010-2014,6040,4267
2015-2018,21708,11409
2019-2021,34153,7563
2022-2023,73742,11487


In [14]:
chi2, p, dof, expected = chi2_contingency(ct)

print("Chi-square:", chi2)
print("df:", dof)
print("p-value:", p)

# Effect size: Cramér's V
n_ct = ct.to_numpy().sum()
r, c = ct.shape
cramers_v = np.sqrt(chi2 / (n_ct * (min(r-1, c-1))))
print("Cramer's V:", cramers_v)

Chi-square: 9489.101971266799
df: 4
p-value: 0.0
Cramer's V: 0.23597793154141616


### Conclusion (Q2 — BEV vs PHEV composition and how it changes by Model Year)

Overall, the regional EV registration population in this dataset is BEV-dominant. The one-sided proportion test for BEV share (H0: p ≤ 0.5 vs H1: p > 0.5) produces an extremely small p-value (it may display as 0 due to numerical precision), so we reject H0 and conclude that BEVs make up a majority of registrations.

The BEV/PHEV mix also changes across model years. Using a chi-square test of independence on Model Year bins (excluding 2024 because the dataset only includes registrations through March 2024), a small p-value leads us to reject the null hypothesis that EV type is independent of model year. This means the BEV share is not constant across model years; newer model-year bins tend to have higher BEV shares than older bins, consistent with a shift toward fully electric vehicles in more recent model years.

## **Question 3 Hypothesis Test**

In [19]:
import math
from statistics import NormalDist

MAKE_COL  = "Make"
MODEL_COL = "Model"

TOP_N_MAKES = 10
P0 = 0.50
ALPHA = 0.05

In [20]:
# Find the most common makes and store the top N make names.
top_makes = df_my[MAKE_COL].value_counts().head(TOP_N_MAKES)
top_make_list = top_makes.index.tolist()
top_makes

Make
TESLA         79471
NISSAN        13984
CHEVROLET     13652
FORD           9177
BMW            7556
KIA            7423
TOYOTA         6255
VOLKSWAGEN     4993
JEEP           4469
HYUNDAI        4399
Name: count, dtype: int64

In [21]:
#Calculate the Top 1 model for each make and proportion
df_top = df_my[df_my[MAKE_COL].isin(top_make_list)].copy()

make_model_counts = (
    df_top.groupby([MAKE_COL, MODEL_COL])
    .size()
    .rename("count")
    .reset_index()
)


top1 = (
    make_model_counts.sort_values([MAKE_COL, "count"], ascending=[True, False])
    .groupby(MAKE_COL)
    .head(1)
    .rename(columns={MODEL_COL: "top_model", "count": "k_top1"})
)

# total of each make
make_total = (
    df_top.groupby(MAKE_COL)
    .size()
    .rename("n_make")
    .reset_index()
)

top1 = top1.merge(make_total, on=MAKE_COL, how="left")
top1["p_hat_top1"] = top1["k_top1"] / top1["n_make"]
top1

,Make,top_model,k_top1,n_make,p_hat_top1
0,BMW,X5,2407,7556,0.318555
1,CHEVROLET,BOLT EV,6812,13652,0.498975
2,FORD,MUSTANG MACH-E,3316,9177,0.361338
3,HYUNDAI,IONIQ 5,2432,4399,0.552853
4,JEEP,WRANGLER,3383,4469,0.756993
5,KIA,NIRO,3144,7423,0.423548
6,NISSAN,LEAF,13352,13984,0.954805
7,TESLA,MODEL Y,35921,79471,0.452001
8,TOYOTA,PRIUS PRIME,2726,6255,0.435811
9,VOLKSWAGEN,ID.4,3928,4993,0.786701


In [22]:
def log_binom_pmf(k, n, p):
    # log( C(n,k) p^k (1-p)^(n-k) )
    return (math.lgamma(n+1) - math.lgamma(k+1) - math.lgamma(n-k+1)
            + k*math.log(p) + (n-k)*math.log(1-p))

def binom_test_greater_exact(k, n, p0):
    # p-value = P(X >= k | n, p0)
    logs = [log_binom_pmf(i, n, p0) for i in range(k, n+1)]
    m = max(logs)
    return float(math.exp(m) * sum(math.exp(li - m) for li in logs))

def wilson_ci(k, n, alpha=0.05):
    z = NormalDist().inv_cdf(1 - alpha/2)
    p_hat = k / n
    denom = 1 + (z**2)/n
    center = (p_hat + (z**2)/(2*n)) / denom
    half = (z * math.sqrt((p_hat*(1-p_hat)/n) + (z**2)/(4*n**2))) / denom
    return float(center - half), float(center + half)

# calculate p-value + CI for each make
pvals = []
ci_lows, ci_highs = [], []

for _, r in top1.iterrows():
    k = int(r["k_top1"])
    n = int(r["n_make"])
    pval = binom_test_greater_exact(k, n, P0)
    ci = wilson_ci(k, n, ALPHA)
    pvals.append(pval)
    ci_lows.append(ci[0])
    ci_highs.append(ci[1])

top1["p_value_exact"] = pvals
top1["ci95_low"] = ci_lows
top1["ci95_high"] = ci_highs

top1.sort_values("p_hat_top1", ascending=False)

,Make,top_model,k_top1,n_make,p_hat_top1,p_value_exact,ci95_low,ci95_high
6,NISSAN,LEAF,13352,13984,0.954805,0.000000e+00,0.951236,0.958125
9,VOLKSWAGEN,ID.4,3928,4993,0.786701,0.000000e+00,0.775121,0.797841
4,JEEP,WRANGLER,3383,4469,0.756993,1.813642e-271,0.744201,0.769343
3,HYUNDAI,IONIQ 5,2432,4399,0.552853,1.259533e-12,0.538120,0.567493
1,CHEVROLET,BOLT EV,6812,13652,0.498975,5.980092e-01,0.490589,0.507361
7,TESLA,MODEL Y,35921,79471,0.452001,1.000000e+00,0.448544,0.455464
8,TOYOTA,PRIUS PRIME,2726,6255,0.435811,1.000000e+00,0.423566,0.448135
5,KIA,NIRO,3144,7423,0.423548,1.000000e+00,0.412350,0.434826
2,FORD,MUSTANG MACH-E,3316,9177,0.361338,1.000000e+00,0.351569,0.371223
0,BMW,X5,2407,7556,0.318555,1.000000e+00,0.308144,0.329150


In [25]:
# Bonferroni test
m = len(top1)
top1["p_value_bonf"] = np.minimum(top1["p_value_exact"] * m, 1.0)
top1["reject_H0_bonf"] = top1["p_value_bonf"] < ALPHA

top1[[MAKE_COL, "top_model", "k_top1", "n_make", "p_hat_top1", "ci95_low", "ci95_high",
      "p_value_exact", "p_value_bonf", "reject_H0_bonf"]].sort_values("p_hat_top1", ascending=False)

,Make,top_model,k_top1,n_make,p_hat_top1,ci95_low,ci95_high,p_value_exact,p_value_bonf,reject_H0_bonf
6,NISSAN,LEAF,13352,13984,0.954805,0.951236,0.958125,0.000000e+00,0.000000e+00,True
9,VOLKSWAGEN,ID.4,3928,4993,0.786701,0.775121,0.797841,0.000000e+00,0.000000e+00,True
4,JEEP,WRANGLER,3383,4469,0.756993,0.744201,0.769343,1.813642e-271,1.813642e-270,True
3,HYUNDAI,IONIQ 5,2432,4399,0.552853,0.538120,0.567493,1.259533e-12,1.259533e-11,True
1,CHEVROLET,BOLT EV,6812,13652,0.498975,0.490589,0.507361,5.980092e-01,1.000000e+00,False
7,TESLA,MODEL Y,35921,79471,0.452001,0.448544,0.455464,1.000000e+00,1.000000e+00,False
8,TOYOTA,PRIUS PRIME,2726,6255,0.435811,0.423566,0.448135,1.000000e+00,1.000000e+00,False
5,KIA,NIRO,3144,7423,0.423548,0.412350,0.434826,1.000000e+00,1.000000e+00,False
2,FORD,MUSTANG MACH-E,3316,9177,0.361338,0.351569,0.371223,1.000000e+00,1.000000e+00,False
0,BMW,X5,2407,7556,0.318555,0.308144,0.329150,1.000000e+00,1.000000e+00,False


**Conclusion**\
The within-make dominance test shows that some brands are overwhelmingly driven by a single flagship model, while others have a more diversified model mix. Nissan is almost entirely driven by the Leaf (top-model share 0.955), Volkswagen is strongly driven by the ID.4 (0.787), and Jeep is strongly driven by the Wrangler (0.757); these top-model shares are far above the 0.50 dominance threshold and are statistically significant (p-values essentially 0). Hyundai is also top-model led, with the IONIQ 5 slightly above majority share (0.553) and still statistically significant. In contrast, Chevrolet (Bolt EV 0.499), Tesla (Model Y 0.452), Toyota (Prius Prime 0.436), Kia (Niro 0.424), Ford (Mustang Mach-E 0.361), and BMW (X5 0.319) do not cross the 0.50 dominance threshold, indicating their registrations are spread across multiple models rather than being dominated by a single model.